# Cosmic Engine - GPU-Accelerated AI Rendering

This notebook provides GPU-accelerated AI rendering for the Cosmic Engine project.

## Workflow:
1. **Local Machine (Phases 1-2)**: Generate L-system geometry (.obj files)
2. **Google Colab (Phase 3)**: AI-synthesize photorealistic images with SDXL

---

### Prerequisites
- Enable GPU: **Runtime > Change runtime type > GPU (T4 or better)**
- Prepare files on local machine:
  ```bash
  ./prepare_colab_upload.sh
  ```
  This creates `cosmic_engine_colab_package.zip` with all necessary files

---

In [ ]:
# Check GPU availability
!nvidia-smi

## Step 1B: Option 1 - Direct Upload to /content/

**You need to upload:**
- All Python files (factory.py, biosim.py, lsystem.py, etc. - about 24 files)
- Sample .obj files (alien.obj, prehistoric.obj, etc.)
- This notebook

**Easiest method:** Upload the zip created by `prepare_colab_upload.sh`

## Step 1A: Choose Your File Upload Method

**Option 1: Direct Upload (Recommended for first time)**
- Files stored in `/content/` (temporary, lost when session ends)
- Faster to set up
- Use cells in Step 1B below

**Option 2: Google Drive**
- Files persist between sessions
- Need to mount Drive first
- Use cells in Step 1C below

Choose ONE option and skip the other!

In [ ]:
# Upload the cosmic_engine_colab_package.zip file
from google.colab import files
print("Please upload cosmic_engine_colab_package.zip")
uploaded = files.upload()

In [ ]:
## Step 3: Install Dependencies with CUDA Support

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("\n✓ Google Drive mounted!")
print("\nSearching for your files...")

# Common locations where users might put files
possible_paths = [
    '/content/drive/MyDrive/cosmic_engine',
    '/content/drive/MyDrive/abop',
    '/content/drive/MyDrive/Colab Notebooks/cosmic_engine',
    '/content/drive/MyDrive/colab_upload',
]

import os
found_path = None
for path in possible_paths:
    if os.path.exists(path) and os.path.exists(f"{path}/factory.py"):
        found_path = path
        print(f"\n✓ Found project files at: {path}")
        break

if not found_path:
    print("\n⚠️  Could not auto-detect project files.")
    print("Please update PROJECT_PATH in the next cell to point to your files.")

## Step 4: Verify CUDA Setup

In [ ]:
# Extract the package
!unzip -q cosmic_engine_colab_package.zip
!ls -la colab_upload/

# Move all files to current directory for easy access
!cp colab_upload/*.py .
!cp colab_upload/*.obj . 2>/dev/null || true
!cp colab_upload/*.ipynb . 2>/dev/null || true

print("\n✓ Files extracted and ready!")
print("\nPython files:")
!ls *.py | head -10
print("\nOBJ files:")
!ls *.obj 2>/dev/null || echo "No .obj files found"

## Step 2: Organize Files (if not already done)

Create proper directory structure:
- `input_geometries/` - for your .obj files
- `output_gallery/` - for generated renders

**Note:** If you used Step 1C (Google Drive), you may have already done this!

In [ ]:
# Install PyTorch with CUDA support
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

# Install core dependencies
!pip install -q numpy numba pandas matplotlib Pillow opencv-python

# Install AI dependencies for Phase 3 (Stable Diffusion XL)
!pip install -q diffusers transformers accelerate xformers

print("✓ All dependencies installed!")

## Step 5: Check Factory Capabilities

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("\n⚠️  WARNING: GPU not detected! Go to Runtime > Change runtime type > GPU")

## Step 6: Generate Analysis Maps (Fast, No GPU)

First, generate depth/normal/AO maps from geometry. This is quick and doesn't require GPU.

In [ ]:
# Create directories
!mkdir -p input_geometries
!mkdir -p output_gallery

# Move .obj files to input_geometries/
!mv *.obj input_geometries/ 2>/dev/null || true

print("Directory structure:")
!ls -lh input_geometries/

print("\n✓ Ready to render!")

## Step 7: AI Image Synthesis (GPU Required)

Now use Stable Diffusion XL to synthesize photorealistic images.

**Note:** First run will download ~7GB of SDXL models. This takes 2-5 minutes.

In [ ]:
# Verify factory.py works
!python factory.py --list-styles

print("\n" + "="*60)
!python factory.py --system-info

## Step 6: Generate Analysis Maps (Fast, No GPU)

First, generate depth/normal/AO maps from geometry. This is quick and doesn't require GPU.

In [ ]:
# Choose your .obj file
INPUT_FILE = "input_geometries/alien.obj"
OUTPUT_DIR = "output_gallery/alien_maps"

!python factory.py --input {INPUT_FILE} --maps-only --output {OUTPUT_DIR}

# Display generated maps
from IPython.display import Image, display
import os

print("\nGenerated Maps:")
for img_file in sorted(os.listdir(OUTPUT_DIR)):
    if img_file.endswith('.png'):
        print(f"\n{img_file}:")
        display(Image(os.path.join(OUTPUT_DIR, img_file), width=512))

## Step 7: AI Image Synthesis (GPU Required)

Now use Stable Diffusion XL to synthesize photorealistic images.

**Note:** First run will download ~7GB of SDXL models. This takes 2-5 minutes.

### 7a. Alien Structure
Negative phototropism + high pipe exponent = spindly, otherworldly forms

In [ ]:
INPUT_FILE = "input_geometries/alien.obj"
OUTPUT_DIR = "output_gallery/alien_renders"
STYLE = "alien"

!python factory.py \
    --input {INPUT_FILE} \
    --style {STYLE} \
    --output {OUTPUT_DIR} \
    --resolution 1024 \
    --samples 4

# Display results
print("\n" + "="*60)
print("RENDERED IMAGES:")
print("="*60)
for img_file in sorted(os.listdir(OUTPUT_DIR)):
    if img_file.endswith('.png') and 'render' in img_file:
        print(f"\n{img_file}:")
        display(Image(os.path.join(OUTPUT_DIR, img_file), width=800))

### 7b. Prehistoric Mega-Flora
Low pipe exponent + sparse branching = thick trunks like Lepidodendron

In [ ]:
INPUT_FILE = "input_geometries/prehistoric.obj"
OUTPUT_DIR = "output_gallery/prehistoric_renders"
STYLE = "prehistoric"

!python factory.py \
    --input {INPUT_FILE} \
    --style {STYLE} \
    --output {OUTPUT_DIR} \
    --resolution 1024 \
    --samples 4

# Display results
print("\n" + "="*60)
print("RENDERED IMAGES:")
print("="*60)
for img_file in sorted(os.listdir(OUTPUT_DIR)):
    if img_file.endswith('.png') and 'render' in img_file:
        print(f"\n{img_file}:")
        display(Image(os.path.join(OUTPUT_DIR, img_file), width=800))

### 7c. Custom Style with Manual Prompt
Create your own aesthetic by writing a custom prompt

In [ ]:
INPUT_FILE = "input_geometries/alien.obj"  # Change to your file
OUTPUT_DIR = "output_gallery/custom_renders"
CUSTOM_PROMPT = "bioluminescent crystalline trees in deep ocean abyss, volumetric god rays, cinematic lighting, 8k"

!python factory.py \
    --input {INPUT_FILE} \
    --prompt "{CUSTOM_PROMPT}" \
    --output {OUTPUT_DIR} \
    --resolution 1024 \
    --samples 2

# Display results
print("\n" + "="*60)
print("RENDERED IMAGES:")
print("="*60)
for img_file in sorted(os.listdir(OUTPUT_DIR)):
    if img_file.endswith('.png') and 'render' in img_file:
        print(f"\n{img_file}:")
        display(Image(os.path.join(OUTPUT_DIR, img_file), width=800))

## Step 8: Batch Processing

Process multiple .obj files with different styles automatically.

In [ ]:
import os

# Process all .obj files with multiple styles
input_dir = "input_geometries"
styles = ["alien", "prehistoric", "bioluminescent"]

for obj_file in sorted(os.listdir(input_dir)):
    if obj_file.endswith('.obj'):
        base_name = obj_file.replace('.obj', '')
        
        for style in styles:
            input_path = os.path.join(input_dir, obj_file)
            output_dir = f"output_gallery/{base_name}_{style}"
            
            print(f"\n{'='*70}")
            print(f"Processing: {obj_file} | Style: {style}")
            print(f"{'='*70}")
            
            !python factory.py \
                --input {input_path} \
                --style {style} \
                --output {output_dir} \
                --resolution 1024 \
                --samples 2

print("\n" + "="*70)
print("✓ Batch processing complete!")
print("="*70)

## Step 9: Download Results

In [ ]:
# Zip all rendered images
!zip -r cosmic_engine_renders.zip output_gallery/

print("\n✓ Created cosmic_engine_renders.zip")
!ls -lh cosmic_engine_renders.zip

# Download to your computer
from google.colab import files
files.download('cosmic_engine_renders.zip')

---

## Tips for Novel Trilogy Development

### Alien Structures (Negative Phototropism)
- **Parameter**: `phototropism_weight = -0.3` to `-0.8` (grows away from light)
- **Parameter**: `pipe_exponent = 2.5` to `3.0` (spindly, top-heavy)
- **Branching**: Asymmetric angles (20°-40° variance)
- **Styles**: `alien`, `bioluminescent`, `crystalline`, `abyssal`

### Prehistoric Mega-Flora (Carboniferous Period)
- **Parameter**: `pipe_exponent = 1.2` to `1.8` (thick trunks)
- **Reference**: Lepidodendron (40m club mosses), Sigillaria
- **Branching**: Sparse, dichotomous (Y-shaped splits)
- **Styles**: `prehistoric`, `botanical`, `fungal`

### Workflow for Best Results
1. **Generate 50-100 candidates locally** (fast, Phase 1-2)
2. **Filter by metrics**: trunk diameter, height/width ratio, branch density
3. **Upload top 10-20 to Colab** for AI rendering
4. **Iterate prompts** to achieve the "uncanny valley" aesthetic
5. **Select 2-3 hero images** per structure type

### Prompt Engineering Tips
- Add **lighting keywords**: `volumetric lighting`, `god rays`, `rim lighting`
- Add **atmosphere**: `misty`, `atmospheric haze`, `dust particles`
- Add **quality tags**: `8k`, `highly detailed`, `cinematic`, `photorealistic`
- Add **negative prompts** (via factory.py): `cartoon, illustration, cgi`

---

## Resources

- **Local Documentation**: `README.md`, `BIOSIM_README.md`, `FACTORY_README.md`
- **L-System Grammar**: See `examples/` directory in source
- **SDXL Model**: https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0
- **Colab GPU Info**: https://research.google.com/colaboratory/faq.html

---

## Troubleshooting

**"python3: can't open file '/content/factory.py': No such file or directory"**
- Your files are in Google Drive, not `/content/`
- Use **Step 1C** (Google Drive option) instead of Step 1B
- Or check your working directory: `!pwd` and `!ls -la`
- The notebook auto-detects Drive paths and copies files to `/content/`

**"ModuleNotFoundError: No module named 'factory'"**
- Make sure you uploaded all .py files to the working directory
- Run `!ls *.py` to verify files are present
- If using Drive, make sure you ran the copy step in Step 1C

**"FileNotFoundError: alien.obj not found"**
- Check files are in `input_geometries/`: `!ls input_geometries/`
- Update INPUT_FILE path to match your actual filename
- If using Drive, verify files were copied: `!ls -la /content/`

**"CUDA out of memory"**
- Lower `--resolution` to 768 or 512
- Reduce `--samples` to 1 or 2
- Run `torch.cuda.empty_cache()` between renders

**"GPU not detected"**
- Go to **Runtime > Change runtime type > Hardware accelerator > GPU**
- Restart runtime and re-run setup cells

**Files disappear after session timeout**
- Colab `/content/` is temporary - lost when session ends
- Use Google Drive (Step 1C) for persistent storage
- Or download results regularly (Step 9)

---

*Cosmic Engine - Colab Edition*  
*For the Novel Trilogy Project*

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy output to Drive
!mkdir -p "/content/drive/MyDrive/cosmic_engine_outputs"
!cp -r output_gallery "/content/drive/MyDrive/cosmic_engine_outputs/"
!cp cosmic_engine_renders.zip "/content/drive/MyDrive/cosmic_engine_outputs/"

print("✓ Saved to Google Drive: MyDrive/cosmic_engine_outputs/")

---

## Advanced: Memory Optimization

If you encounter OOM (Out of Memory) errors:

In [ ]:
# Strategy 1: Lower resolution
!python factory.py --input input_geometries/alien.obj --style alien --resolution 768

# Strategy 2: Fewer samples
!python factory.py --input input_geometries/alien.obj --style alien --samples 1

# Strategy 3: Clear GPU cache between runs
import torch
torch.cuda.empty_cache()
!nvidia-smi

# Strategy 4: Use CPU rendering (SLOW but works)
# !python factory.py --input input.obj --style alien --device cpu

---

## Tips for Novel Trilogy Development

### Alien Structures (Negative Phototropism)
- **Parameter**: `phototropism_weight = -0.3` to `-0.8` (grows away from light)
- **Parameter**: `pipe_exponent = 2.5` to `3.0` (spindly, top-heavy)
- **Branching**: Asymmetric angles (20°-40° variance)
- **Styles**: `alien`, `bioluminescent`, `crystalline`, `deep_ocean`

### Prehistoric Mega-Flora (Carboniferous Period)
- **Parameter**: `pipe_exponent = 1.2` to `1.8` (thick trunks)
- **Reference**: Lepidodendron (40m club mosses), Sigillaria
- **Branching**: Sparse,dichotomous (Y-shaped splits)
- **Styles**: `prehistoric`, `carboniferous`, `swamp`, `ancient_forest`

### Workflow for Best Results
1. **Generate 50-100 candidates locally** (fast, Phase 1-2)
2. **Filter by metrics**: trunk diameter, height/width ratio, branch density
3. **Upload top 10-20 to Colab** for AI rendering
4. **Iterate prompts** to achieve the "uncanny valley" aesthetic
5. **Select 2-3 hero images** per structure type

### Prompt Engineering Tips
- Add **lighting keywords**: `volumetric lighting`, `god rays`, `rim lighting`
- Add **atmosphere**: `misty`, `atmospheric haze`, `dust particles`
- Add **quality tags**: `8k`, `highly detailed`, `cinematic`, `photorealistic`
- Add **negative prompts** (via factory.py): `cartoon, illustration, cgi`

---

## Resources

- **Local Documentation**: `README.md`, `BIOSIM_README.md`, `FACTORY_README.md`
- **L-System Grammar**: See `examples/` directory in source
- **SDXL Model**: https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0
- **Colab GPU Info**: https://research.google.com/colaboratory/faq.html

---

## Troubleshooting

**"ModuleNotFoundError: No module named 'factory'"**
- Make sure you uploaded all .py files to the working directory
- Run `!ls *.py` to verify files are present

**"FileNotFoundError: alien.obj not found"**
- Check files are in `input_geometries/`: `!ls input_geometries/`
- Update INPUT_FILE path to match your actual filename

**"CUDA out of memory"**
- Lower `--resolution` to 768 or 512
- Reduce `--samples` to 1 or 2
- Run `torch.cuda.empty_cache()` between renders

**"GPU not detected"**
- Go to **Runtime > Change runtime type > Hardware accelerator > GPU**
- Restart runtime and re-run setup cells

---

*Cosmic Engine - Colab Edition*  
*For the Novel Trilogy Project*